In [11]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models
from qdrant_client.models import VectorParams, Distance, PointStruct
from tqdm import tqdm
import os 
from FlagEmbedding import BGEM3FlagModel 

In [2]:
import torch, gc, os
os.environ["TOKENIZERS_PARALLELISM"] = "false" # Prevents the most common deadlock
gc.collect()
torch.cuda.empty_cache()

In [3]:
import os
import torch

# Force clear any specific GPU selection variables
if "CUDA_VISIBLE_DEVICES" in os.environ:
    del os.environ["CUDA_VISIBLE_DEVICES"]

# Check if PyTorch can see the GPU again
print(f"Is CUDA available? {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Is CUDA available? True
GPU: NVIDIA GeForce RTX 3060


In [4]:
pq_path  = "../arxiv_filtered_master.parquet"

In [5]:
collection_name = "arxiv_papers"
batch_size = 256

In [6]:
print("Load embedding moedel")
model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True, devices=['cuda:0'])

Load embedding moedel


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

In [7]:

client = QdrantClient(url="http://localhost:6333")

In [8]:
model_dim = 1024

In [9]:
if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config={
            "dense": models.VectorParams(
                size = model_dim,
                distance = models.Distance.COSINE
            )
        },
        sparse_vectors_config={
            "sparse": models.SparseVectorParams(
                index=models.SparseIndexParams(
                    on_disk=True,)
            )
        }
    )


ResponseHandlingException: [Errno 111] Connection refused

In [ ]:
df = pd.read_parquet(pq_path, columns = ['title','abstract','id','year'])

In [ ]:
df.shape

In [ ]:
total_batches = len(df) // batch_size + 1
print(f"Indexing {len(df)} papers with BGE-M3...")

In [ ]:
df.head()

In [ ]:
import hashlib
def stable_hash_to_int(string_id):
     hash_obj = hashlib.sha256(string_id.encode('utf-8'))
     return int(hash_obj.hexdigest(), 16) % (2**63) #fits 64 bit signed integer ,bit masking to fit into qdrant limit

In [ ]:
for i in tqdm(range(0,len(df), batch_size)):
    batch = df.iloc[i:i+batch_size]

    #no search doc prefix for bgem3 
    texts = [f"{row['title']}\n{row['abstract']}" for _,row in batch.iterrows()]

    #bgem3 dense by default 
    output = model.encode(
        texts,
        batch_size=batch_size,
        max_length = 512,
        return_dense = True,
        return_sparse = True,
        return_colbert_vecs = False,
    )

    points = []

    for idx,(_,row) in enumerate(batch.iterrows()):

        dense_vec = output['dense_vecs'][idx]
        
        sparse_weight_dict = output['lexical_weights'][idx]
        sparse_indices = [ int(k) for k in sparse_weight_dict.keys()]
        sparse_values = [float(v) for v in sparse_weight_dict.values()]
    
        point_id = stable_hash_to_int(str(row['id']))
    
        points.append(models.PointStruct(
            id = point_id,
            vector = {
                "dense": dense_vec.tolist(),
                "sparse": models.SparseVector(
                    indices = sparse_indices,
                    values = sparse_values
                )
            },
            payload={
                'id' : str(row['id']),
                'title' : row['title'],
                'abstract' : row['abstract'],
                'year': row['year']
            }
        ))

    client.upsert(collection_name= collection_name, points=points,)

print("Hybrid Indexing Complete")

    
        


In [ ]:
collection_info = client.get_collection(collection_name=collection_name)
print(f"Total points indexed: {collection_info.points_count}")

# Detailed check
count_result = client.count(collection_name=collection_name, exact=True)
print(f"Confirmed Count: {count_result.count}")

In [ ]:
records, _ = client.scroll(
    collection_name=collection_name,
    limit=5,             # Just look at the last 5
    with_payload=True,   # YES, show me the title/summary
    with_vectors=False   # NO, don't show me the long lists of numbers
)

print("\n--- Verifying Latest Entries ---")
for rec in records:
    print(f"ID: {rec.id}")
    print(f"Title: {rec.payload.get('title')}")
    print(f"Summary Snippet: {rec.payload.get('abstract')[:100]}...")
    print("-" * 30)

In [ ]:
target_arxiv_id = "2309.06794"
target_point_id = stable_hash_to_int(target_arxiv_id)

# Try to retrieve specifically this paper
result = client.retrieve(
    collection_name="arxiv_papers",
    ids=[target_point_id],
    with_payload=True
)

if result:
    print(f"✅ Found it! Title: {result[0].payload['title']}")
else:
    print(f"❌ Not found. Current Point Count in DB: {client.count(collection_name='arxiv_papers').count}")

In [ ]:
from qdrant_client import models

client.create_payload_index(
    collection_name="arxiv_papers",
    field_name="year",
    field_schema=models.PayloadSchemaType.INTEGER,
)
print("Year index created! 🚀")

In [ ]:
hy